In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
!pip install pandas librosa soundfile

import os
import pandas as pd
import librosa
import soundfile as sf

data_path = "/content/drive/MyDrive/Processed_Patient_Audio"
save_root = "/content/drive/MyDrive/Segmented_Data"
os.makedirs(save_root, exist_ok=True)

dataset = []

MIN_DUR = 2.0   # minimum 2 sec
MAX_DUR = 5.0   # maximum 5 sec

for root, _, files in os.walk(data_path):

    if "patient.wav" in files:

        print(f"\nProcessing: {root}")

        audio_path = os.path.join(root, "patient.wav")
        csv_path = root.replace("Processed_Patient_Audio", "Dementia_Data") + "/turn_level.csv"

        if not os.path.exists(csv_path):
            continue

        df = pd.read_csv(csv_path)
        df = df[df["speaker"] == "SPEAKER_00"]

        y, sr = librosa.load(audio_path, sr=16000)

        seg_id = 0
        buffer_text = ""
        start_time = None

        for i, row in df.iterrows():

            if start_time is None:
                start_time = row["start"]

            buffer_text += " " + str(row["text"])
            end_time = row["end"]

            duration = end_time - start_time

            # 🔥 KEEP MERGING until at least MIN_DUR
            if duration < MIN_DUR:
                continue

            # 🔥 LIMIT to MAX_DUR
            if duration > MAX_DUR:
                end_time = start_time + MAX_DUR

            # EXTRACT AUDIO
            start_sample = int(start_time * sr)
            end_sample = int(end_time * sr)

            segment_audio = y[start_sample:end_sample]

            # SAFETY (very rare but important)
            if len(segment_audio) == 0:
                start_time = None
                buffer_text = ""
                continue

            # SAVE
            save_dir = root.replace("Processed_Patient_Audio", "Segmented_Data")
            os.makedirs(save_dir, exist_ok=True)

            file_path = os.path.join(save_dir, f"seg_{seg_id}.wav")
            sf.write(file_path, segment_audio, sr)

            dataset.append({
                "audio": file_path,
                "text": buffer_text.strip()
            })

            seg_id += 1
            buffer_text = ""
            start_time = None

        # 🔥 HANDLE LEFTOVER BUFFER (VERY IMPORTANT)
        if start_time is not None:
            end_time = row["end"]

            start_sample = int(start_time * sr)
            end_sample = int(end_time * sr)

            segment_audio = y[start_sample:end_sample]

            if len(segment_audio) > 0:
                file_path = os.path.join(save_dir, f"seg_{seg_id}_last.wav")
                sf.write(file_path, segment_audio, sr)

                dataset.append({
                    "audio": file_path,
                    "text": buffer_text.strip()
                })


Processing: /content/drive/MyDrive/Processed_Patient_Audio/dementia/cookie/processed_audios/001-0

Processing: /content/drive/MyDrive/Processed_Patient_Audio/dementia/cookie/processed_audios/001-2

Processing: /content/drive/MyDrive/Processed_Patient_Audio/dementia/cookie/processed_audios/003-0

Processing: /content/drive/MyDrive/Processed_Patient_Audio/dementia/cookie/processed_audios/005-0

Processing: /content/drive/MyDrive/Processed_Patient_Audio/dementia/cookie/processed_audios/005-2

Processing: /content/drive/MyDrive/Processed_Patient_Audio/dementia/cookie/processed_audios/007-1

Processing: /content/drive/MyDrive/Processed_Patient_Audio/dementia/cookie/processed_audios/007-3

Processing: /content/drive/MyDrive/Processed_Patient_Audio/dementia/cookie/processed_audios/010-0

Processing: /content/drive/MyDrive/Processed_Patient_Audio/dementia/cookie/processed_audios/010-1

Processing: /content/drive/MyDrive/Processed_Patient_Audio/dementia/cookie/processed_audios/010-2

Processin

In [10]:
df_out = pd.DataFrame(dataset)
df_out.to_csv("/content/drive/MyDrive/Segmented_Data/final_segments.csv", index=False)

print("✅ SEGMENTATION WITH MERGING DONE")

✅ SEGMENTATION WITH MERGING DONE
